# 列表推导式和函数式辅助工具
- 推导式不止列表推导式，但列表推导式可以覆盖大多数的场景

| 数据流 | 常用写法 | 结果 |
|---|---|---|
| 映射 | 列表推导式、`map()` | 多个新元素 |
| 筛选 | 带 `if` 的列表推导式、`filter()` | 部分原元素 |
| 聚合 | `sum()` | 一个合计值 |
| 提供短函数 | `lambda` | 函数对象 |

## 列表推导式
### 列表推导式（循环-转换）
- 列表推导式创建一个新列表，依次读取输入元素，计算表达式，并把结果放入新列表
- 比普通 for 循环更高效、更轻量、更可读
- 列表推导式会立即执行并建立完整列表;输入量很大时，它会阻塞

In [ ]:
# 同义表达对比
result = []
for item in iterable:
    result.append(expression)

result = [expression for item in iterable]

In [3]:
raw_fields = [
    " CustomEr_ID ",
    " PHONE      ",
    " CreATed_At ",
]

standardized_fields = [field.strip().lower() for field in raw_fields]
print(standardized_fields)

['customer_id', 'phone', 'created_at']


### 带条件的列表推导式（循环-过滤-转换）
- 在列表推导式的基础上，每轮先判断条件
    - 条件为真才计算输出表达式并添加结果
    - 条件为假时，该元素不进入新列表

In [ ]:
# 对比
result = []
for item in iterable:
    if condition:
        result.append(expression)

result = [expression for item in iterable if condition]

In [4]:
raw_fields = [
    " Customer_ID ",
    "",
    " PHONE ",
    "   ",
    " Created_At ",
]

standardized_fields = [
    field.strip().lower() # 如果去空格后仍为truthy值，进行处理并添加到列表
    for field in raw_fields
    if field.strip() # 如果去两侧空格后为falsy值（空字符串），直接排除
]

print(standardized_fields)

['customer_id', 'phone', 'created_at']


- 条件写在末尾时才表示过滤，写在前面属于 `expression`
- 即末尾的 `if` 才决定元素是否进入输出；前面的表达式决定进入输出后变成什么

In [5]:
booleans_fields = [
    "敏感字段" if field == "phone" else "普通字段"
    for field in standardized_fields
]
print(booleans_fields)

['普通字段', '敏感字段', '普通字段']


### 列表推导式的使用边界
1. 列表推导式把循环、筛选、转换压缩到一个表达式中
2. 压缩只改变表达形式，不会让底层判断和转换步骤消失
3. 如果需要进行复杂流程控制（捕获异常、多步状态更新等），不适合使用推导式
    - 简单映射、简单筛选适合推导式
    - 复杂状态处理使用普通循环

## filter()：根据谓词保留原元素
1. `filter(predicate, iterable)` 逐项调用判断函数（谓词函数）
    - 函数结果为真时，保留原元素
    - 结果为假时，丢弃原元素
2. filter()只看函数返回值的truthiess，可以通过建立bool_map来实现业务逻辑
3. filter()输出的是原元素，与谓词无关
4. filter() 返回惰性迭代器，不立即返回列表
    - 需要时再去消费，迭代器被消费后，不能从头重复使用



In [6]:
rules = [
    {"rule_id": "DQ-01", "active": "Active"},
    {"rule_id": "DQ-02", "active": "Inactive"},
    {"rule_id": "DQ-03", "active": "Pending"},
]

bool_map = {
    "Active":True,
    "Inactive":False,
    "Pending":False
} # 建立符合业务逻辑的布尔映射
rule_filter = filter(lambda rule:bool_map[rule["active"]],rules)
active_rules = list(rule_filter)
print(active_rules) # 返回rules中符合谓词判断的rule
print(list(rule_filter)) # filter迭代器被消费后无法再次消费

[{'rule_id': 'DQ-01', 'active': 'Active'}]
[]


## map()：把每个元素转换为新值
- `map(function, iterable)` 逐项调用转换函数，并产生每次函数调用的返回值，返回值的类型可以与输入完全不同
- map() 也返回惰性迭代器，不立即返回列表
    - 需要时再去消费，迭代器被消费后，不能从头重复使用

In [7]:
raw_rule_ids = [
    " dq-01 ",
    "DQ-02",
    " dq-03",
]

standardized_iterator = map(
    lambda rule_id: rule_id.strip().upper(),
    raw_rule_ids,
)

standardized_rule_ids = list(standardized_iterator)
print(standardized_rule_ids)
print(list(standardized_iterator))

['DQ-01', 'DQ-02', 'DQ-03']
[]


## 惰性工具流水线
- 多个惰性工具可以连接成流水线；外层消费结果时，数据才逐项穿过每一层
- 先过滤，再映射：map(transform, filter(predicate, iterable))
- 先映射，再过滤：filter(predicate,map(transform,iterable))

In [8]:
raw_amounts = ["10", "", "5"]

non_empty_filter = filter(
    lambda value: value != "", # 接收可迭代对象返回的元素，返回元素与空字符串的不等判断结果
    raw_amounts,
)

amounts_map = map(
    float,
    non_empty_filter, # 接收上游过滤器返回的元素，返回转换成float
)

result = list(amounts_map)

print(result)

[10.0, 5.0]


## sum()：把多个数值折叠成一个结果
- `sum(iterable, start=0)` 逐项执行加法，最后返回一个合计值
- 可以通过start参数设置累计的初始值

In [9]:
raw_amounts = ["10", "", "5"]

total_amount = sum(
    map(
        float,
        filter(lambda value: value != "", raw_amounts)
    )
)

print(total_amount)

15.0


- 输入元素必须能与累计值执行加法
- 当输入为空时，默认返回start值
- 浮点数合计可能受到二进制浮点精度影响；精确财务口径需要使用适当的数值类型和舍入规则

## lambda：创建短小匿名函数
- lambda 表达式创建函数对象，`lambda parameters: expression`
- 函数被传给 map()、filter() 或排序工具后，工具在需要时调用它

In [ ]:
# 对比
def function(parameters):
    return expression(parameters)

lambda parameters: expression # 等价于一个只返回表达式结果的简单函数
# 表达式可以用推导式，更加灵活地实现业务逻辑

- lambda 的函数体只能是一个表达式，不能直接包含普通赋值、for 语句、while 语句或多条语句
- lambda 只负责创建函数；何时调用、给它什么元素以及如何消费返回值，由外部调用者决定